# STERIS Predictive Maintenance on Snowflake
## From SCADA Telemetry & eMaint Work Orders → RUL Predictions

> **DEMO SCRIPT — Cell 0**
>
> **SAY:** *"Perdita, today I'm going to show you a complete predictive maintenance pipeline — from your raw Ignition SCADA telemetry and eMaint work orders all the way to a prioritized maintenance work queue with RUL predictions, failure mode identification, and component-level diagnostics. The entire thing runs inside Snowflake — no external ML platform, no data movement."*
>
> **HIGHLIGHT:** Point to the table below — each row maps to a section of the notebook.
>
> **PERDITA — Set the hook:** *"Everything you asked about — prognostics with specific timeframes, failure identification down to the component, SCADA integration, CMMS history — we'll address each one. And the key differentiator: the data never leaves the platform."*

| Section | What We Show | Perdita's Requirement |
|---|---|---|
| 1. Source Data | SCADA sensor streams + eMaint CMMS failure history | SCADA Integration + CMMS Historical Records |
| 2. Data Cleansing | Null handling, outlier detection, quality validation | Data readiness for ML |
| 3. Feature Engineering | Rolling statistics, z-scores, CMMS cumulative metrics | Raw data → learnable patterns |
| 4. Model Training | XGBoost Regression (RUL) + Classification (Failure Mode) | Prognostics & RUL + Failure Identification |
| 5. Predictions | Per-asset RUL, component-at-risk, prioritized work queue | Actionable maintenance decisions |

In [ ]:
USE ROLE SF_INTELLIGENCE_DEMO;
USE DATABASE STERIS_RELIABILITY_DB;
USE WAREHOUSE STERIS_ANALYTICS_WH;

---
## 1. Source Data: What Feeds the Model

> **DEMO SCRIPT — Cell 2**
>
> **SAY:** *"Let's start with the data. In most organizations, SCADA telemetry lives in one system and CMMS work orders live in another. Engineers export CSVs, email spreadsheets, manually reconcile. Here, both streams land in one governed platform."*
>
> **HIGHLIGHT:** The two bullet points below — emphasize the scale (175K readings) and that both sources are queryable side-by-side.
>
> **PERDITA — SCADA Integration requirement:** *"This is your Ignition SCADA data — vibration, temperature, motor current — alongside your eMaint failure records. No ETL pipeline stitching them together. They're just... here."*

- **SCADA Telemetry** (Ignition) — 175K hourly sensor readings across 20 assets
- **CMMS Failure Records** (eMaint) — 19 documented failure events with root cause

### 1a. Asset Fleet

In [ ]:
SELECT
    ASSET_ID,
    ASSET_NAME,
    ASSET_TYPE,
    MANUFACTURER || ' ' || MODEL AS EQUIPMENT,
    INSTALL_DATE,
    DATEDIFF('year', INSTALL_DATE, CURRENT_DATE()) || ' yrs' AS AGE,
    CRITICALITY_SCORE,
    '$' || TO_CHAR(PRODUCTION_IMPACT_HOURLY_USD, '999,999') AS HOURLY_IMPACT
FROM RAW.ASSET_MASTER
ORDER BY CRITICALITY_SCORE DESC
LIMIT 10;

### 1b. SCADA Telemetry Stream

> **DEMO SCRIPT — Cell 4-6**
>
> **SAY:** *"Here's the raw SCADA feed — hourly vibration, temperature, and motor current from every asset. 175,000 readings, sitting in Snowflake, ready to query."*
>
> **HIGHLIGHT:** Point to the row count (175K) and the three sensor types. When you run Cell 6, emphasize that it shows readings per asset — this proves data completeness.
>
> **AHA MOMENT:** *"Notice we're querying this with plain SQL. No special connector, no API call to Ignition. The data landed here via standard ingestion and now it's just a table."*

Hourly sensor readings from Ignition SCADA — vibration (mm/s), motor temperature (°C), motor current (A), ambient temperature. In Snowflake, this data is queryable the moment it lands — no staging databases or ETL pipelines between ingestion and analysis.

In [ ]:
SELECT
    ASSET_ID,
    READING_TIMESTAMP,
    ROUND(VIBRATION_MM_S, 3) AS VIBRATION_MM_S,
    ROUND(MOTOR_TEMP_C, 1) AS MOTOR_TEMP_C,
    ROUND(MOTOR_CURRENT_A, 2) AS MOTOR_CURRENT_A,
    ROUND(AMBIENT_TEMP_C, 1) AS AMBIENT_TEMP_C,
    CYCLE_COUNT
FROM RAW.SENSOR_READINGS_GENERATED
WHERE ASSET_ID = 'AST-010'
ORDER BY READING_TIMESTAMP DESC
LIMIT 10;

In [ ]:
SELECT
    COUNT(*) AS TOTAL_READINGS,
    COUNT(DISTINCT ASSET_ID) AS ASSETS,
    MIN(READING_TIMESTAMP)::DATE AS EARLIEST,
    MAX(READING_TIMESTAMP)::DATE AS LATEST,
    ROUND(COUNT(*) / COUNT(DISTINCT ASSET_ID)) AS AVG_READINGS_PER_ASSET
FROM RAW.SENSOR_READINGS_GENERATED;

### 1c. eMaint CMMS — Failure History

> **DEMO SCRIPT — Cell 7-9**
>
> **SAY:** *"And here's the other half — your eMaint failure records. 19 documented failures across 5 assets with root cause, downtime hours, and repair details."*
>
> **HIGHLIGHT:** Run Cell 8 to show the failure events. Point to FAILURE_TYPE and COMPONENT_FAILED columns — these become the supervised learning labels.
>
> **PERDITA — CMMS Historical Records requirement:** *"This is exactly what you described — using historical maintenance records to train the models. The failure type tells the model WHAT failed, the timestamp tells it WHEN, and the repair hours tell it HOW LONG it took. That's your ground truth."*
>
> **WOW MOMENT (Cell 9 — downtime cost):** *"And here's the business case — we can calculate the actual cost of unplanned downtime per asset using the hourly production impact. This is what we'll use later to prioritize the work queue."*

Every documented failure event with root cause, downtime, and repair cost. Because this lives in the same Snowflake account as the SCADA data, we can JOIN sensor patterns to failure outcomes in a single SQL statement — this is the foundation of supervised ML training.

In [ ]:
SELECT
    EVENT_ID,
    ASSET_ID,
    FAILURE_TIMESTAMP::DATE AS FAILURE_DATE,
    FAILURE_TYPE,
    ROOT_CAUSE,
    ROUND(DOWNTIME_HOURS, 1) AS DOWNTIME_HRS,
    '$' || TO_CHAR(REPAIR_COST_USD, '999,999') AS REPAIR_COST
FROM RAW.FAILURE_EVENTS
ORDER BY FAILURE_TIMESTAMP;

In [ ]:
SELECT
    FAILURE_TYPE,
    COUNT(*) AS OCCURRENCES,
    ROUND(AVG(DOWNTIME_HOURS), 1) AS AVG_DOWNTIME_HRS,
    ROUND(AVG(REPAIR_COST_USD), 0) AS AVG_REPAIR_COST,
    LISTAGG(DISTINCT ASSET_ID, ', ') AS AFFECTED_ASSETS
FROM RAW.FAILURE_EVENTS
GROUP BY FAILURE_TYPE
ORDER BY OCCURRENCES DESC;

---
## 2. Data Cleansing & Preparation

> **DEMO SCRIPT — Cells 10-13**
>
> **SAY:** *"Before we build features, we need to validate the data. Are there missing readings? Outliers? Gaps in coverage? With Snowflake, this is pure SQL against the live data — no exporting to pandas for profiling."*
>
> **HIGHLIGHT:** Run Cell 11 (null check) — point out zero nulls. Run Cell 12 (outlier detection) — explain z-scores: *"A z-score above 3 means a reading is more than 3 standard deviations from the mean. These aren't errors — they're degradation signals."*
>
> **CRITICAL FOR PERDITA:** *"This is important — in most setups, data scientists would export this to a Jupyter notebook on their laptop to profile it. Here, the profiling runs where the data lives. No copy, no export, no version mismatch."*

In [ ]:
SELECT
    COUNT(*) AS TOTAL_ROWS,
    SUM(CASE WHEN VIBRATION_MM_S IS NULL THEN 1 ELSE 0 END) AS NULL_VIBRATION,
    SUM(CASE WHEN MOTOR_TEMP_C IS NULL THEN 1 ELSE 0 END) AS NULL_TEMP,
    SUM(CASE WHEN MOTOR_CURRENT_A IS NULL THEN 1 ELSE 0 END) AS NULL_CURRENT,
    ROUND(100.0 * SUM(CASE WHEN VIBRATION_MM_S IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS NULL_PCT_VIB,
    ROUND(AVG(VIBRATION_MM_S), 4) AS MEAN_VIB,
    ROUND(STDDEV(VIBRATION_MM_S), 4) AS STD_VIB,
    ROUND(MIN(VIBRATION_MM_S), 4) AS MIN_VIB,
    ROUND(MAX(VIBRATION_MM_S), 4) AS MAX_VIB
FROM RAW.SENSOR_READINGS_GENERATED;

In [ ]:
WITH stats AS (
    SELECT
        ASSET_ID,
        AVG(VIBRATION_MM_S) AS MEAN_VIB,
        STDDEV(VIBRATION_MM_S) AS STD_VIB,
        AVG(MOTOR_TEMP_C) AS MEAN_TEMP,
        STDDEV(MOTOR_TEMP_C) AS STD_TEMP
    FROM RAW.SENSOR_READINGS_GENERATED
    GROUP BY ASSET_ID
)
SELECT
    s.ASSET_ID,
    COUNT(*) AS TOTAL_READINGS,
    SUM(CASE WHEN ABS(r.VIBRATION_MM_S - s.MEAN_VIB) > 3 * s.STD_VIB THEN 1 ELSE 0 END) AS VIBRATION_OUTLIERS,
    SUM(CASE WHEN ABS(r.MOTOR_TEMP_C - s.MEAN_TEMP) > 3 * s.STD_TEMP THEN 1 ELSE 0 END) AS TEMP_OUTLIERS,
    ROUND(100.0 * SUM(CASE WHEN ABS(r.VIBRATION_MM_S - s.MEAN_VIB) > 3 * s.STD_VIB THEN 1 ELSE 0 END) / COUNT(*), 2) AS OUTLIER_PCT
FROM RAW.SENSOR_READINGS_GENERATED r
JOIN stats s ON r.ASSET_ID = s.ASSET_ID
GROUP BY s.ASSET_ID
HAVING OUTLIER_PCT > 0 OR OUTLIER_PCT = 0
ORDER BY OUTLIER_PCT DESC;

In [ ]:
SELECT
    ASSET_ID,
    COUNT(DISTINCT READING_TIMESTAMP::DATE) AS DAYS_WITH_DATA,
    COUNT(*) AS TOTAL_READINGS,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT READING_TIMESTAMP::DATE), 0), 1) AS READINGS_PER_DAY,
    MIN(READING_TIMESTAMP)::DATE AS FIRST_READING,
    MAX(READING_TIMESTAMP)::DATE AS LAST_READING
FROM RAW.SENSOR_READINGS_GENERATED
GROUP BY ASSET_ID
ORDER BY ASSET_ID
LIMIT 10;

**Data Quality Summary:** Sensor data is complete with no nulls. Outliers correlate with assets approaching failure — these are real degradation signals, not noise. We retain them for modeling.

---
## 3. Feature Engineering

> **DEMO SCRIPT — Cells 14-17**
>
> **SAY:** *"This is where raw readings become learnable patterns. We take 175,000 hourly sensor readings and compute 24 engineered features using SQL window functions — rolling averages, trend percentages, z-scores, and CMMS cumulative metrics."*
>
> **HIGHLIGHT:** Point to the 4 feature categories below. When you run Cell 15 (the CREATE VIEW), say: *"Watch this — 24 features computed over 175K rows, and it runs in seconds on elastic compute. No Spark cluster to provision."*
>
> **AHA MOMENT (Cell 16 — the specific example):** *"Let me show you a specific asset. AST-002 on October 14th: vibration is 0.78 mm/s daily but the 7-day average is 0.75 — and the trend is UP 5.4%. The z-score is 2.39, meaning it's approaching statistical anomaly. It's had 3 corrective work orders, and it's been 14 days since the last one. THESE are the patterns the model learns from."*
>
> **PERDITA — This is your 'aha' moment:** *"This is exactly what your team described — taking SCADA telemetry and CMMS history and turning them into prognostic features. The vibration trend tells you it's getting worse. The z-score tells you how abnormal it is. The CMMS features tell you it has a history. Together, they predict WHEN and WHAT will fail."*

**24 features** in 4 categories:
- **Sensor Daily Stats:** avg, max, min, stddev for vibration/temp/current
- **Rolling Windows:** 7-day and 30-day moving averages and maxima
- **Trend & Anomaly:** 7d/30d trend percentages, z-scores
- **CMMS History:** cumulative corrective WOs, downtime hours, MTTR, days since last corrective

In [ ]:
CREATE OR REPLACE VIEW FEATURES.VW_ML_TRAINING_FEATURES AS
WITH daily_sensor AS (
    SELECT
        ASSET_ID,
        DATE_TRUNC('DAY', READING_TIMESTAMP)::DATE AS FEATURE_DATE,
        AVG(VIBRATION_MM_S) AS VIBRATION_DAILY_AVG,
        MAX(VIBRATION_MM_S) AS VIBRATION_DAILY_MAX,
        MIN(VIBRATION_MM_S) AS VIBRATION_DAILY_MIN,
        STDDEV(VIBRATION_MM_S) AS VIBRATION_DAILY_STD,
        AVG(MOTOR_TEMP_C) AS MOTOR_TEMP_DAILY_AVG,
        MAX(MOTOR_TEMP_C) AS MOTOR_TEMP_DAILY_MAX,
        AVG(MOTOR_CURRENT_A) AS MOTOR_CURRENT_DAILY_AVG,
        MAX(MOTOR_CURRENT_A) AS MOTOR_CURRENT_DAILY_MAX,
        AVG(AMBIENT_TEMP_C) AS AMBIENT_TEMP_DAILY_AVG,
        COUNT(*) AS READINGS_PER_DAY
    FROM RAW.SENSOR_READINGS_GENERATED
    GROUP BY ASSET_ID, DATE_TRUNC('DAY', READING_TIMESTAMP)::DATE
),
rolling_features AS (
    SELECT
        ds.*,
        AVG(ds.VIBRATION_DAILY_AVG) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS VIBRATION_7D_AVG,
        MAX(ds.VIBRATION_DAILY_MAX) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS VIBRATION_7D_MAX,
        AVG(ds.VIBRATION_DAILY_AVG) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ) AS VIBRATION_30D_AVG,
        (ds.VIBRATION_DAILY_AVG - LAG(ds.VIBRATION_DAILY_AVG, 7) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE
        )) / NULLIF(LAG(ds.VIBRATION_DAILY_AVG, 7) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE
        ), 0) * 100 AS VIBRATION_TREND_7D,
        (ds.VIBRATION_DAILY_AVG - LAG(ds.VIBRATION_DAILY_AVG, 30) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE
        )) / NULLIF(LAG(ds.VIBRATION_DAILY_AVG, 30) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE
        ), 0) * 100 AS VIBRATION_TREND_30D,
        (ds.MOTOR_TEMP_DAILY_AVG - LAG(ds.MOTOR_TEMP_DAILY_AVG, 7) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE
        )) / NULLIF(LAG(ds.MOTOR_TEMP_DAILY_AVG, 7) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE
        ), 0) * 100 AS TEMP_TREND_7D,
        (ds.MOTOR_CURRENT_DAILY_AVG - LAG(ds.MOTOR_CURRENT_DAILY_AVG, 7) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE
        )) / NULLIF(LAG(ds.MOTOR_CURRENT_DAILY_AVG, 7) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE
        ), 0) * 100 AS CURRENT_TREND_7D,
        (ds.VIBRATION_DAILY_AVG - AVG(ds.VIBRATION_DAILY_AVG) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        )) / NULLIF(STDDEV(ds.VIBRATION_DAILY_AVG) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ), 0) AS VIBRATION_Z_SCORE,
        (ds.MOTOR_TEMP_DAILY_AVG - AVG(ds.MOTOR_TEMP_DAILY_AVG) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        )) / NULLIF(STDDEV(ds.MOTOR_TEMP_DAILY_AVG) OVER (
            PARTITION BY ds.ASSET_ID ORDER BY ds.FEATURE_DATE ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ), 0) AS TEMP_Z_SCORE
    FROM daily_sensor ds
),
asset_features AS (
    SELECT
        rf.*,
        DATEDIFF('DAY', am.INSTALL_DATE, rf.FEATURE_DATE) AS ASSET_AGE_DAYS,
        DATEDIFF('DAY', am.LAST_MAINTENANCE_DATE, rf.FEATURE_DATE) AS DAYS_SINCE_LAST_MAINTENANCE,
        am.CRITICALITY_SCORE,
        am.PRODUCTION_IMPACT_HOURLY_USD,
        am.EXPECTED_LIFE_YEARS
    FROM rolling_features rf
    JOIN RAW.ASSET_MASTER am ON rf.ASSET_ID = am.ASSET_ID
),
cmms_agg AS (
    SELECT
        af.ASSET_ID,
        af.FEATURE_DATE,
        COUNT(fe.EVENT_ID) AS CUMULATIVE_CORRECTIVE_WOS,
        COALESCE(SUM(fe.DOWNTIME_HOURS), 0) AS CUMULATIVE_DOWNTIME_HOURS,
        COALESCE(AVG(fe.DOWNTIME_HOURS), 0) AS AVG_MTTR_HOURS,
        COALESCE(DATEDIFF('DAY', MAX(fe.FAILURE_TIMESTAMP::DATE), af.FEATURE_DATE), 365) AS DAYS_SINCE_LAST_CORRECTIVE
    FROM asset_features af
    LEFT JOIN RAW.FAILURE_EVENTS fe
        ON fe.ASSET_ID = af.ASSET_ID
        AND fe.FAILURE_TIMESTAMP::DATE <= af.FEATURE_DATE
    GROUP BY af.ASSET_ID, af.FEATURE_DATE
)
SELECT
    af.*,
    COALESCE(ca.CUMULATIVE_CORRECTIVE_WOS, 0) AS CUMULATIVE_CORRECTIVE_WOS,
    COALESCE(ca.CUMULATIVE_DOWNTIME_HOURS, 0) AS CUMULATIVE_DOWNTIME_HOURS,
    COALESCE(ca.AVG_MTTR_HOURS, 0) AS AVG_MTTR_HOURS,
    COALESCE(ca.DAYS_SINCE_LAST_CORRECTIVE, 365) AS DAYS_SINCE_LAST_CORRECTIVE
FROM asset_features af
LEFT JOIN cmms_agg ca
    ON af.ASSET_ID = ca.ASSET_ID
    AND af.FEATURE_DATE = ca.FEATURE_DATE;

In [ ]:
SELECT
    ASSET_ID,
    FEATURE_DATE,
    ROUND(VIBRATION_DAILY_AVG, 3) AS VIB_AVG,
    ROUND(VIBRATION_7D_AVG, 3) AS VIB_7D,
    ROUND(VIBRATION_30D_AVG, 3) AS VIB_30D,
    ROUND(VIBRATION_TREND_7D, 1) AS VIB_TREND_7D_PCT,
    ROUND(VIBRATION_Z_SCORE, 2) AS VIB_Z,
    ROUND(MOTOR_TEMP_DAILY_AVG, 1) AS TEMP_AVG,
    ROUND(TEMP_TREND_7D, 1) AS TEMP_TREND_PCT,
    CUMULATIVE_CORRECTIVE_WOS AS CMMS_WOS,
    ROUND(AVG_MTTR_HOURS, 1) AS AVG_MTTR,
    DAYS_SINCE_LAST_CORRECTIVE AS DAYS_LAST_CORR,
    ASSET_AGE_DAYS,
    CRITICALITY_SCORE
FROM FEATURES.VW_ML_TRAINING_FEATURES
WHERE ASSET_ID = 'AST-002'
AND FEATURE_DATE BETWEEN '2024-10-01' AND '2024-10-20'
ORDER BY FEATURE_DATE DESC;

In [ ]:
SELECT
    COUNT(*) AS TOTAL_FEATURE_ROWS,
    COUNT(DISTINCT ASSET_ID) AS ASSETS,
    MIN(FEATURE_DATE) AS EARLIEST,
    MAX(FEATURE_DATE) AS LATEST,
    SUM(CASE WHEN VIBRATION_7D_AVG IS NOT NULL THEN 1 ELSE 0 END) AS ROWS_WITH_ROLLING
FROM FEATURES.VW_ML_TRAINING_FEATURES;

### Feature Engineering Summary

> **DEMO SCRIPT — Cell 18**
>
> **SAY:** *"Each row is now one asset-day with 24 features. And because these are Snowflake views, they auto-refresh as new sensor data arrives. There's no batch ETL to re-run."*
>
> **ANTICIPATE THIS QUESTION:** If Perdita asks *"Why not use Snowflake's native Feature Store?"* — say: *"Great question. The Feature Store is ideal for production deployments with versioned feature sets and point-in-time lookups. For this demo, SQL views give us the same auto-refresh behavior with simpler setup. In production, we'd likely migrate these views into the Feature Store for version control and time-travel capabilities."*

### 3b. Labeled Dataset — Joining Features to Known Failures

> **DEMO SCRIPT — Cells 19-20**
>
> **SAY:** *"Now we create our training labels. For every feature-day within 90 days of a known failure, we know exactly what failed and how many days until it happened. This JOIN is one SQL statement — because both datasets live in the same platform."*
>
> **HIGHLIGHT (Cell 20):** Point to the DAYS_TO_FAILURE column — *"This is the RUL target. The model learns: 'when features look like THIS, failure is X days away.'"*
>
> **CRITICAL FOR PERDITA:** *"Notice the gap assignment logic in Cell 19 — when an asset has multiple failures, we assign each feature-day to the NEXT upcoming failure only. This prevents mislabeling between failure events. That's an ML best practice that's easy to implement when your data is all in one place."*

In [ ]:
CREATE OR REPLACE VIEW FEATURES.VW_ML_LABELED_DATASET AS
SELECT
    f.*,
    fe.FAILURE_TYPE AS FAILURE_MODE,
    DATEDIFF('DAY', f.FEATURE_DATE, fe.FAILURE_TIMESTAMP::DATE) AS DAYS_TO_FAILURE,
    CASE WHEN DATEDIFF('DAY', f.FEATURE_DATE, fe.FAILURE_TIMESTAMP::DATE) <= 30 THEN 1 ELSE 0 END AS FAILURE_WITHIN_30D,
    CASE WHEN DATEDIFF('DAY', f.FEATURE_DATE, fe.FAILURE_TIMESTAMP::DATE) <= 14 THEN 1 ELSE 0 END AS FAILURE_WITHIN_14D
FROM FEATURES.VW_ML_TRAINING_FEATURES f
INNER JOIN (
    SELECT
        ASSET_ID,
        FAILURE_TIMESTAMP,
        FAILURE_TYPE,
        LAG(FAILURE_TIMESTAMP) OVER (PARTITION BY ASSET_ID ORDER BY FAILURE_TIMESTAMP) AS PREV_FAILURE
    FROM RAW.FAILURE_EVENTS
) fe ON f.ASSET_ID = fe.ASSET_ID
    AND f.FEATURE_DATE < fe.FAILURE_TIMESTAMP::DATE
    AND f.FEATURE_DATE >= COALESCE(fe.PREV_FAILURE::DATE, '2024-01-01')
    AND DATEDIFF('DAY', f.FEATURE_DATE, fe.FAILURE_TIMESTAMP::DATE) <= 90
WHERE f.FEATURE_DATE >= '2024-02-01';

In [ ]:
SELECT
    FAILURE_MODE,
    COUNT(*) AS SAMPLES,
    ROUND(AVG(DAYS_TO_FAILURE), 1) AS AVG_DAYS_TO_FAIL,
    MIN(DAYS_TO_FAILURE) AS MIN_DTF,
    MAX(DAYS_TO_FAILURE) AS MAX_DTF
FROM FEATURES.VW_ML_LABELED_DATASET
WHERE VIBRATION_7D_AVG IS NOT NULL
GROUP BY FAILURE_MODE
ORDER BY SAMPLES DESC;

---
## 4. Model Training

> **DEMO SCRIPT — Cells 21-27**
>
> **SAY:** *"Now for the ML. Two models, both trained right here in this notebook using Snowpark Python. The data never leaves Snowflake."*
>
> **HIGHLIGHT:** The table below — two models answering two different questions.
>
> **PERDITA — Prognostics & RUL requirement:** *"The regressor answers 'WHEN will it fail?' — that's your RUL with specific timeframes. The classifier answers 'WHAT will fail and WHY?' — that's your failure identification down to the component."*
>
> **WOW MOMENT (Cell 22):** *"Notice — I'm importing XGBoost, scikit-learn, and pandas right here in the same notebook where we just wrote SQL. SQL for feature engineering, Python for model training, same session, same governance. That's the Snowflake advantage."*

| Model | Target | Perdita's Question |
|---|---|---|
| **RUL Regressor** | DAYS_TO_FAILURE (continuous) | *"When will this asset fail?"* |
| **Failure Classifier** | FAILURE_MODE (4 classes) | *"What component will fail and why?"* |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from snowflake.snowpark.context import get_active_session
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score, classification_report
from sklearn.ensemble import IsolationForest
import xgboost as xgb

session = get_active_session()
print('Libraries loaded.')

In [ ]:
FEATURE_COLS = [
    'VIBRATION_DAILY_AVG', 'VIBRATION_DAILY_MAX', 'VIBRATION_DAILY_MIN', 'VIBRATION_DAILY_STD',
    'MOTOR_TEMP_DAILY_AVG', 'MOTOR_TEMP_DAILY_MAX',
    'MOTOR_CURRENT_DAILY_AVG', 'MOTOR_CURRENT_DAILY_MAX',
    'VIBRATION_7D_AVG', 'VIBRATION_7D_MAX', 'VIBRATION_30D_AVG',
    'VIBRATION_TREND_7D', 'VIBRATION_TREND_30D', 'TEMP_TREND_7D', 'CURRENT_TREND_7D',
    'VIBRATION_Z_SCORE', 'TEMP_Z_SCORE',
    'ASSET_AGE_DAYS', 'DAYS_SINCE_LAST_MAINTENANCE', 'CRITICALITY_SCORE',
    'CUMULATIVE_CORRECTIVE_WOS', 'CUMULATIVE_DOWNTIME_HOURS', 'AVG_MTTR_HOURS', 'DAYS_SINCE_LAST_CORRECTIVE',
]

failure_df = session.sql(f"""
    SELECT ASSET_ID, FEATURE_DATE, {', '.join(FEATURE_COLS)}, FAILURE_MODE, DAYS_TO_FAILURE
    FROM FEATURES.VW_ML_LABELED_DATASET
    WHERE VIBRATION_7D_AVG IS NOT NULL
    ORDER BY ASSET_ID, FEATURE_DATE
""").to_pandas()

healthy_df = session.sql(f"""
    SELECT f.ASSET_ID, f.FEATURE_DATE, {', '.join(['f.' + c for c in FEATURE_COLS])},
           'HEALTHY' AS FAILURE_MODE, 365 AS DAYS_TO_FAILURE
    FROM FEATURES.VW_ML_TRAINING_FEATURES f
    LEFT JOIN RAW.FAILURE_EVENTS fe
        ON f.ASSET_ID = fe.ASSET_ID
        AND ABS(DATEDIFF('DAY', f.FEATURE_DATE, fe.FAILURE_TIMESTAMP::DATE)) <= 45
    WHERE fe.ASSET_ID IS NULL AND f.VIBRATION_7D_AVG IS NOT NULL AND f.FEATURE_DATE >= '2024-02-01'
    ORDER BY RANDOM()
    LIMIT {len(failure_df)}
""").to_pandas()

df = pd.concat([failure_df, healthy_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Training dataset: {len(df):,} samples ({len(failure_df):,} failure + {len(healthy_df):,} healthy)')
print(f'\nFailure mode distribution:')
print(df['FAILURE_MODE'].value_counts().to_string())

### 4a. XGBoost Regression — Remaining Useful Life (Days to Failure)

> **DEMO SCRIPT — Cell 25**
>
> **SAY:** *"First model: RUL regression. We're training XGBoost on 2,500 samples — balanced between failure and healthy readings — to predict days until failure."*
>
> **HIGHLIGHT THE RESULTS:** *"MAE of 1.83 days means on average, the prediction is off by less than 2 days. R² of 0.98 means the model explains 98% of the variance in time-to-failure. For maintenance planning, that's extremely actionable."*
>
> **PERDITA — This is her #1 requirement:** *"This is exactly what you asked for — prognostics with specific timeframes. Not a risk score, not a traffic light. An actual number of days until failure, accurate to within 2 days."*
>
> **ANTICIPATE:** If she asks about the balanced training (1,261 + 1,261), explain: *"We balance healthy and failure samples equally so the model doesn't just learn to predict 'healthy' for everything — which is what 95% of the data actually is."*

In [ ]:
rul_df = df[df['FAILURE_MODE'] != 'HEALTHY'].copy()
rul_df['DAYS_TO_FAILURE'] = rul_df['DAYS_TO_FAILURE'].clip(upper=90)

X = rul_df[FEATURE_COLS].fillna(0).astype(float)
y = rul_df['DAYS_TO_FAILURE'].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rul_model = xgb.XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42, objective='reg:squarederror'
)
rul_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = rul_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'RUL Regression Results:')
print(f'  Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'  MAE:   {mae:.2f} days')
print(f'  R\u00b2:    {r2:.4f}')

importance = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': rul_model.feature_importances_})
importance = importance.sort_values('Importance', ascending=False)
print(f'\nTop 10 Features Driving RUL Prediction:')
for _, row in importance.head(10).iterrows():
    bar = chr(9608) * int(row['Importance'] * 50)
    print(f'  {row["Feature"]:<35} {row["Importance"]:.4f} {bar}')

### 4b. XGBoost Classification — Failure Mode Identification

> **DEMO SCRIPT — Cell 27**
>
> **SAY:** *"Second model: failure mode classification. Same 24 features, but now predicting WHICH type of failure is approaching — bearing wear, bracket loose, motor overload, or electrical fault. Each maps to a specific component."*
>
> **HIGHLIGHT THE RESULTS:** *"80% accuracy with a time-based train/test split — we train on Jan-Aug failures and test on Sep-Dec. This is honest ML: the model has never seen the test data's time period, just like production."*
>
> **PERDITA — Failure Identification requirement:** *"This answers your second big question: not just WHEN but WHAT. Instead of 'check AST-010,' it's 'the motor bearing on AST-010 is degrading.' That's the difference between a site visit and a targeted repair."*
>
> **ANTICIPATE:** If she asks about the 80% vs higher accuracy: *"We deliberately use a time-based split to prevent data leakage. A random split gave 100% — too perfect, because the model was memorizing time windows instead of learning sensor patterns. 80% on truly unseen future data is a strong result, and it improves as you collect more failure events."*
>
> **KEY INSIGHT:** *"BEARING_WEAR has perfect recall because we have 681 training samples. ELECTRICAL_FAULT has lower representation (87 samples from one asset). More failure data = better classification — which is another reason to consolidate in Snowflake."*

In [ ]:
cls_df = df[df['FAILURE_MODE'] != 'HEALTHY'].copy()
le = LabelEncoder()
cls_df['LABEL'] = le.fit_transform(cls_df['FAILURE_MODE'])

X = cls_df[FEATURE_COLS].fillna(0).astype(float)
y = cls_df['LABEL']

from datetime import date
train_mask = cls_df['FEATURE_DATE'] < date(2024, 9, 1)
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

cls_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    random_state=42, objective='multi:softprob',
    num_class=len(le.classes_), eval_metric='mlogloss'
)
cls_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = cls_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print(f'Failure Mode Classification Results (time-based split: train < Sep, test >= Sep):')
print(f'  Classes:  {list(le.classes_)}')
print(f'  Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'  Accuracy: {acc:.4f}')
print(f'  F1:       {f1:.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=le.classes_, labels=range(len(le.classes_)))}')

---
## 5. Predictions — Addressing Perdita's Requirements

> **DEMO SCRIPT — Cell 29**
>
> **SAY:** *"Now we score the entire fleet. Both models run against every asset's latest sensor features — one Python call, results in seconds."*
>
> **WOW MOMENT — Pause on the output table:** *"Look at this. Every asset in your fleet, ranked by urgency. AST-004 — motor bearing, 7 days. AST-015 — control board, 7 days but 78% confidence — lower confidence because we had fewer electrical fault training examples. AST-001 — motor overload, 73 days out, plenty of time to plan."*
>
> **PERDITA — All four requirements in one table:**

| Column | Her Requirement | What It Shows |
|---|---|---|
| **RUL_DAYS** | Prognostics with specific timeframes | Exact days until predicted failure |
| **FAILURE_MODE** | Failure identification | What type of failure is approaching |
| **COMPONENT** | Component-level diagnostics | Which part to inspect/replace |
| **CONFIDENCE** | Trust calibration | How certain the model is (reflects training data volume) |
| **FAIL DATE** | Maintenance scheduling | Calendar date for work order planning |

> **SAY:** *"This is the shift from reactive to predictive. Instead of waiting for AST-004 to fail and scrambling, you schedule the bearing replacement next week."*

In [ ]:
import json
from datetime import datetime

latest = session.sql(f"""
    SELECT ASSET_ID, FEATURE_DATE, {', '.join(FEATURE_COLS)}
    FROM FEATURES.VW_ML_TRAINING_FEATURES
    WHERE FEATURE_DATE = (SELECT MAX(FEATURE_DATE) FROM FEATURES.VW_ML_TRAINING_FEATURES)
    AND VIBRATION_7D_AVG IS NOT NULL
    ORDER BY ASSET_ID
""").to_pandas()

X_score = latest[FEATURE_COLS].fillna(0).astype(float)

rul_preds = np.clip(rul_model.predict(X_score), 7, 365)
fm_encoded = cls_model.predict(X_score)
fm_proba = cls_model.predict_proba(X_score)
failure_modes = le.inverse_transform(fm_encoded)
confidence = np.max(fm_proba, axis=1)

FAILURE_TO_COMPONENT = {
    'BEARING_WEAR': 'MOTOR_BEARING', 'BRACKET_LOOSE': 'MOUNTING_BRACKET',
    'MOTOR_OVERLOAD': 'DRIVE_MOTOR', 'ELECTRICAL_FAULT': 'CONTROL_BOARD',
}
FAILURE_REASONS = {
    'BEARING_WEAR': 'Vibration signature indicates progressive bearing degradation consistent with rolling element fatigue.',
    'BRACKET_LOOSE': 'Intermittent vibration spikes with increasing baseline suggest loosening mounting hardware.',
    'MOTOR_OVERLOAD': 'Motor current trending above rated capacity with thermal rise indicating insulation breakdown.',
    'ELECTRICAL_FAULT': 'Erratic sensor patterns with temperature correlation indicate control board degradation.',
}

top_feats = sorted(zip(FEATURE_COLS, rul_model.feature_importances_), key=lambda x: x[1], reverse=True)[:5]

records = []
for i, row in latest.iterrows():
    fm = failure_modes[i]
    records.append({
        'ASSET_ID': row['ASSET_ID'],
        'COMPONENT': FAILURE_TO_COMPONENT.get(fm, 'UNKNOWN'),
        'RUL_DAYS': round(float(rul_preds[i]), 1),
        'RUL_HOURS': int(rul_preds[i] * 24),
        'CONFIDENCE_PCT': round(float(confidence[i]) * 100, 1),
        'FAILURE_MODE': fm,
        'FAILURE_REASON': FAILURE_REASONS.get(fm, 'Under investigation'),
        'PREDICTED_FAILURE_DATE': (datetime.now() + pd.Timedelta(days=float(rul_preds[i]))).strftime('%Y-%m-%d'),
        'ANOMALY_SCORE': round(float(row.get('VIBRATION_Z_SCORE', 0) or 0) / 3.0, 4),
        'TOP_CONTRIBUTING_FEATURES': json.dumps([{'feature': f, 'importance': round(float(imp), 4)} for f, imp in top_feats]),
    })

result_df = pd.DataFrame(records).sort_values('RUL_DAYS')

print(f'Fleet Predictions ({len(result_df)} assets):\n')
print(f'{"ASSET":<10} {"COMPONENT":<18} {"RUL":>6} {"MODE":<20} {"CONF":>5} {"FAIL DATE":>12}')
print('-' * 80)
for _, r in result_df.iterrows():
    print(f"{r['ASSET_ID']:<10} {r['COMPONENT']:<18} {r['RUL_DAYS']:>5.0f}d {r['FAILURE_MODE']:<20} {r['CONFIDENCE_PCT']:>4.0f}% {r['PREDICTED_FAILURE_DATE']:>12}")

### 5b. Write Predictions to Snowflake

> **DEMO SCRIPT — Cell 31**
>
> **SAY:** *"Now the predictions land as governed Snowflake tables. Two tables: the predictions themselves, and an evidence table documenting WHY each prediction was made — the SCADA readings and CMMS records behind it."*
>
> **HIGHLIGHT:** The two table names — `ML.COMPONENT_RUL_PREDICTIONS` (20 rows) and `ML.PREDICTION_EVIDENCE` (80 rows, 4 per asset).
>
> **PERDITA — Operational readiness:** *"These are immediately queryable. Your Tableau dashboard, your Power BI report, your eMaint integration — they can all read from this table right now. No file export, no API. Same RBAC that governs the source data governs the predictions."*

In [ ]:
pred_spdf = session.create_dataframe(records)
pred_spdf.write.mode('overwrite').save_as_table('ML.COMPONENT_RUL_PREDICTIONS')

evidence_records = []
for i, row in latest.iterrows():
    asset_id = row['ASSET_ID']
    vib = row.get('VIBRATION_DAILY_AVG', 0) or 0
    vib_7d = row.get('VIBRATION_7D_AVG', 0) or 0
    vib_trend = row.get('VIBRATION_TREND_7D', 0) or 0
    vib_z = row.get('VIBRATION_Z_SCORE', 0) or 0
    temp = row.get('MOTOR_TEMP_DAILY_AVG', 0) or 0
    current = row.get('MOTOR_CURRENT_DAILY_AVG', 0) or 0
    cum_wo = row.get('CUMULATIVE_CORRECTIVE_WOS', 0) or 0
    avg_mttr = row.get('AVG_MTTR_HOURS', 0) or 0
    days_corr = row.get('DAYS_SINCE_LAST_CORRECTIVE', 365) or 365

    evidence_records.append({'ASSET_ID': asset_id, 'EVIDENCE_TYPE': 'SCADA_VIBRATION',
        'EVIDENCE_DETAIL': f'Vibration: {vib:.2f} mm/s (7d avg: {vib_7d:.2f}). Trend: {vib_trend:+.1f}%. Z-score: {vib_z:.2f}.',
        'DATA_SOURCE': 'IGNITION_SCADA'})
    evidence_records.append({'ASSET_ID': asset_id, 'EVIDENCE_TYPE': 'SCADA_TEMPERATURE',
        'EVIDENCE_DETAIL': f'Motor temp: {temp:.1f} C. Motor current: {current:.1f}A.',
        'DATA_SOURCE': 'IGNITION_SCADA'})
    evidence_records.append({'ASSET_ID': asset_id, 'EVIDENCE_TYPE': 'CMMS_HISTORY',
        'EVIDENCE_DETAIL': f'{int(cum_wo)} corrective WOs. Avg MTTR: {avg_mttr:.1f} hrs. Last corrective: {int(days_corr)} days ago.',
        'DATA_SOURCE': 'EMAINT_CMMS'})
    evidence_records.append({'ASSET_ID': asset_id, 'EVIDENCE_TYPE': 'FAILURE_PATTERN_MATCH',
        'EVIDENCE_DETAIL': f'{failure_modes[i]} pattern matched with {confidence[i]:.0%} confidence.',
        'DATA_SOURCE': 'ML_MODEL'})

ev_spdf = session.create_dataframe(evidence_records)
ev_spdf.write.mode('overwrite').save_as_table('ML.PREDICTION_EVIDENCE')

print(f'Written to Snowflake:')
print(f'  ML.COMPONENT_RUL_PREDICTIONS: {len(records)} rows')
print(f'  ML.PREDICTION_EVIDENCE: {len(evidence_records)} rows')

### 5c. High-Risk Assets — Drill Down

> **DEMO SCRIPT — Cell 33**
>
> **SAY:** *"Now watch this — a maintenance supervisor who knows zero Python can write this SQL query: 'Show me every asset failing within 30 days, sorted by urgency.' That's it. ML outputs are just tables."*
>
> **AHA MOMENT:** *"The person who built the model and the person who acts on the predictions don't need to use the same tools. Data scientist works in Python, maintenance planner works in SQL or Tableau. Same data, same governance."*

In [ ]:
SELECT
    p.ASSET_ID,
    a.ASSET_NAME,
    p.COMPONENT,
    p.RUL_DAYS || ' days' AS TIME_TO_FAILURE,
    p.CONFIDENCE_PCT || '%' AS CONFIDENCE,
    p.FAILURE_MODE,
    p.FAILURE_REASON,
    p.PREDICTED_FAILURE_DATE
FROM ML.COMPONENT_RUL_PREDICTIONS p
JOIN RAW.ASSET_MASTER a ON p.ASSET_ID = a.ASSET_ID
WHERE p.RUL_DAYS <= 30
ORDER BY p.RUL_DAYS ASC;

### 5d. Prediction Evidence — SCADA + CMMS Traceability

> **DEMO SCRIPT — Cell 35**
>
> **SAY:** *"Here's what sets this apart from a black-box ML tool. Every prediction has evidence: the specific vibration reading, the temperature, the CMMS work order history that drove it."*
>
> **HIGHLIGHT:** Point to the 4 evidence rows for AST-010 — SCADA_VIBRATION, SCADA_TEMPERATURE, CMMS_HISTORY, FAILURE_PATTERN_MATCH. Each traces back to a source system.
>
> **PERDITA — Trust and adoption:** *"Your maintenance team won't act on a number they can't explain. This evidence table means any technician can ask 'why does the model think this bearing is failing?' and get a concrete answer: 'because vibration is trending up 5%, it's had 3 corrective WOs, and the pattern matches bearing wear with 94% confidence.' That's how you get adoption."*

In [ ]:
SELECT
    EVIDENCE_TYPE,
    EVIDENCE_DETAIL,
    DATA_SOURCE
FROM ML.PREDICTION_EVIDENCE
WHERE ASSET_ID = 'AST-010'
ORDER BY EVIDENCE_TYPE;

### 5e. Maintenance Work Queue — Prioritized by RUL

> **DEMO SCRIPT — Cell 37 — THIS IS THE GRAND FINALE**
>
> **SAY:** *"And here's the deliverable. A prioritized maintenance work queue — sorted by urgency, with the predicted failure mode, component at risk, confidence level, and the dollar cost of downtime per hour."*
>
> **WOW MOMENT — Linger on this output:** *"AST-004: motor bearing, 7 days, URGENT, $4,500/hour production impact. AST-001: motor overload, 73 days, LOW priority. Your team now knows exactly what to schedule, when, and what it costs if they don't."*
>
> **PERDITA — The close:** *"This is the bridge from data science to operations. This query can feed your eMaint system directly. It can be a Tableau dashboard. It can be a daily email. And it updates every time the models re-score — because the features auto-refresh from live SCADA data."*
>
> **KEY POINT:** *"Everything you just saw — from raw sensor data to this work queue — runs inside Snowflake. One platform, one governance model, no data movement. That's the value proposition."*

In [ ]:
SELECT
    p.ASSET_ID,
    a.ASSET_NAME,
    p.COMPONENT,
    p.RUL_DAYS || ' days' AS TIME_TO_FAILURE,
    p.FAILURE_MODE,
    p.CONFIDENCE_PCT || '%' AS CONFIDENCE,
    p.PREDICTED_FAILURE_DATE,
    a.CRITICALITY_SCORE,
    '$' || TO_CHAR(a.PRODUCTION_IMPACT_HOURLY_USD, '999,999') AS HOURLY_IMPACT,
    CASE
        WHEN p.RUL_DAYS <= 14 THEN 'URGENT'
        WHEN p.RUL_DAYS <= 30 THEN 'HIGH'
        WHEN p.RUL_DAYS <= 60 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS PRIORITY
FROM ML.COMPONENT_RUL_PREDICTIONS p
JOIN RAW.ASSET_MASTER a ON p.ASSET_ID = a.ASSET_ID
ORDER BY p.RUL_DAYS ASC;

---
## Summary

> **DEMO SCRIPT — Closing**
>
> **SAY:** *"Let me tie this back to what you asked for."*

| Perdita's Requirement | What We Just Showed | Key Cell |
|---|---|---|
| **Prognostics & RUL** | XGBoost regression → MAE 1.83 days, R² 0.98 | Cell 25 |
| **Failure Identification** | XGBoost classifier → failure mode → component, 80% accuracy | Cell 27 |
| **SCADA Integration** | 175K Ignition readings → 24 features → predictions | Cells 15-16 |
| **CMMS Historical Records** | eMaint failure events + cumulative WOs as model inputs | Cells 8, 19 |

> **SAY — The Snowflake differentiators:**
> 1. *"Zero data movement — SCADA, CMMS, features, models, and predictions all in one platform"*
> 2. *"SQL + Python in one notebook — feature engineering in SQL, model training in Python, same governance"*
> 3. *"Elastic compute — warehouse scales up for training, scales down after. Pay only for what you use"*
> 4. *"Operational readiness — predictions are tables. Any BI tool, dashboard, or CMMS can query them today"*
> 5. *"Full traceability — every prediction traces back to the SCADA reading and CMMS record that drove it"*

> **CLOSING STATEMENT:**
>
> **SAY:** *"The shift is this:"*
> ```
> BEFORE: "AST-010 has high risk"
>         (opaque score, external tool, data exported somewhere)
>
>  AFTER: "AST-010 motor bearing will fail in 11 days because vibration
>          is trending up and the asset has 5 corrective WOs on record"
>         (explainable, traceable, governed, actionable — all in Snowflake)
> ```
> *"That's what we can build together. Questions?"*